# CSC 381/576 · Day 2 — ETL on a genuinely messy file

**Module 1 · Getting Data In** &nbsp;·&nbsp; Aug 26, 2026

One file, two acts:

- **Act 1** — load it, look at it, fix the types
- **Act 2** — clean the text, decide about the hard cases, save the result

The dataset is 44 used-car listings, typed by humans, exported from three systems.
Nobody cleaned it. That is the point.

---

### Rule zero: the raw file is read-only

We never write to `data/raw/`. We read it once, copy it, and every change we make lives
in the code below. If you overwrite the raw file you have destroyed the only copy of the
truth and you can no longer check your own work.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import openai

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)

print('pandas', pd.__version__)


ModuleNotFoundError: No module named 'pandas'

### Finding the file

This cell works in two places without editing it:

- **VS Code / local**, with the project laid out as `csc381-cars/notebooks/` and `csc381-cars/data/raw/`
- **Colab**, where you upload `used_cars_raw.csv` next to the notebook (the folder icon on the left → upload)

In [ ]:
RAW = Path('../data/raw/used_cars_raw.csv')      # local project layout
if not RAW.exists():
    RAW = Path('used_cars_raw.csv')              # Colab: uploaded next to the notebook

raw = pd.read_csv(RAW)
df  = raw.copy()          # raw stays pristine; df is the one we mutate

print(RAW.resolve())
print(df.shape)


### A helper we will use all night

Every time we remove rows we print how many. This is not politeness — it is the single
habit that separates a defensible analysis from a broken one.

In [ ]:
def rows_changed(before, after, what):
    """Print how many rows an operation removed, and complain if it was a lot."""
    lost = before - after
    pct  = 100 * lost / before if before else 0
    print(f'{what}: {before} -> {after} rows  ({lost} removed, {pct:.1f}%)')
    return after


---

# Act 1 — Load it, look at it, fix the types

## 1.1  First look

Three calls, in this order, every single time you meet a new file.

Watch the `Dtype` column in `.info()`. How many columns are `object` that should not be?

In [ ]:
df.head(10)


In [ ]:
df.info()


In [ ]:
df.dtypes


### Try to average the price

Run this and read the error out loud. This is what an `object` column costs you.

In [ ]:
df['price'].mean()        # <- uncomment in class. It will not do what you want.


## 1.2  The row that is not a car

The last row of the file is a spreadsheet footer that survived the export. It is not an
observation, so it must not be in an observations table.

Notice we identify it by a *rule*, not by row number — row numbers change, rules do not.

In [ ]:
df[df['listing_id'] == 'TOTAL']


In [ ]:
before = len(df)
df = df[df['listing_id'] != 'TOTAL'].copy()
rows_changed(before, len(df), 'drop the TOTAL footer row')


## 1.3  price → a real number

`$12,500`, `14750`, and `11,900.00` are three ways of writing a number. Strip everything
that is not a digit or a dot, then convert.

`errors='coerce'` turns anything that still will not convert into `NaN` instead of
raising — but you must then go and *look* at what became NaN.

In [ ]:
price_clean = (df['price']
               .astype('string')
               .str.replace(r'[$,]', '', regex=True)
               .str.strip())

df['price_usd'] = pd.to_numeric(price_clean, errors='coerce')

print('failed to convert:', df['price_usd'].isna().sum())
df[['listing_id', 'price', 'price_usd']].head(8)


In [ ]:
df['price_usd'].describe()


Look at that `min`. We will come back to it in Act 2 — it is one of tonight's judgment calls.

## 1.4  mileage → a number *and* a unit

`78,000 mi` is two variables jammed into one cell: a quantity and a unit. Tidy data says
one column, one variable — so we split it into two.

**Watch what happens to the rows that have no unit at all.**

In [ ]:
df['mileage_value'] = pd.to_numeric(
    df['mileage'].astype('string').str.replace(r'[^0-9.\-]', '', regex=True),
    errors='coerce')

df['mileage_unit'] = df['mileage'].astype('string').str.extract(r'(km|mi)', expand=False)

df[['listing_id', 'mileage', 'mileage_value', 'mileage_unit']].head(8)


In [ ]:
# Which rows have a number but no unit? Do NOT skip this check.
df.loc[df['mileage_unit'].isna(), ['listing_id', 'mileage', 'mileage_value']]


Two rows have a bare negative number and no unit. That is a second defect hiding behind
the first one — and you only found it because you checked instead of assuming.

Now convert everything to one unit. We assume a missing unit means miles, because every
labelled row in this file that is not `km` is `mi`. **That assumption is a decision, so we
write it down.**

In [ ]:
KM_TO_MI = 0.621371

# .fillna(False) matters: rows with no unit give NA, and NA cannot steer np.where
is_km = df['mileage_unit'].eq('km').fillna(False).astype(bool)

df['mileage_mi'] = np.where(is_km, df['mileage_value'] * KM_TO_MI, df['mileage_value'])
df['mileage_mi'] = pd.to_numeric(df['mileage_mi'], errors='coerce').round(0)

print('rows converted from km:', int(is_km.sum()))
df.loc[is_km, ['listing_id', 'mileage', 'mileage_value', 'mileage_mi']]


## 1.5  Three date formats → one datetime column

`2026-08-01`, `08/03/2026`, and `Aug 5, 2026` are all in this column. `format='mixed'`
lets pandas work each row out individually.

Then check: **did anything fail?** A silent NaT is worse than an error.

In [ ]:
df['listed_on'] = pd.to_datetime(df['date_listed'], format='mixed')

print('failed to parse:', df['listed_on'].isna().sum())
print(df['listed_on'].min(), '->', df['listed_on'].max())
df[['listing_id', 'date_listed', 'listed_on']].head(8)


Now that it is a real datetime, date arithmetic works — which it never does on strings.

In [ ]:
df['days_listed'] = (pd.Timestamp('2026-08-26') - df['listed_on']).dt.days
df[['listing_id', 'listed_on', 'days_listed']].head(5)


## 1.6  year — where the impossible values live

One row says `'17`. Two say years that cannot exist. We fix the fixable one and *flag*
the rest rather than quietly deleting them.

In [ ]:
year_clean = (df['year']
              .astype('string')
              .str.strip()
              .str.replace(r"^'(\d{2})$", r'20\1', regex=True))   # '17 -> 2017

df['year_num'] = pd.to_numeric(year_clean, errors='coerce').astype('Int64')

df.loc[~df['year_num'].between(1990, 2027), ['listing_id', 'year', 'year_num']]


### End of Act 1 — check where we are

In [ ]:
df.info()


**The question to ask out loud:** how many columns are still `object` that should not be?

---

# Act 2 — Clean the text, decide the hard cases, save

Act 1 was mechanical: there is a right answer and code can find it. Act 2 is not.
Everything below this line involves a decision that you have to make and defend.

## 2.1  make — five spellings, one car brand

`value_counts(dropna=False)` is the best cleaning tool in pandas. Run it *before* you
clean and *again* after. If the number of distinct values did not drop, you cleaned nothing.

In [ ]:
df['make'].value_counts(dropna=False)


In [ ]:
df['make_clean'] = df['make'].astype('string').str.strip().str.title()
df['make_clean'].value_counts(dropna=False)


`strip()` and `title()` fixed the whitespace and the casing for free.

They did **not** fix `Toyata`, and they did not fix `Chevy` vs `Chevrolet`. Those are
judgment calls: is `Toyata` a typo for Toyota, or a brand you have never heard of?

We decide explicitly, in a mapping we own and can point at later.

In [ ]:
MAKE_FIXES = {
    'Toyata': 'Toyota',      # typo — same models, same price range as the Toyotas
    'Chevy':  'Chevrolet',   # informal name for the same manufacturer
}

df['make_clean'] = df['make_clean'].replace(MAKE_FIXES)
df['make_clean'].value_counts(dropna=False)


## 2.2  transmission — five encodings, two values

In [ ]:
df['transmission'].value_counts(dropna=False)


In [ ]:
TRANSMISSION = {
    'Automatic': 'Automatic', 'auto': 'Automatic', 'A': 'Automatic',
    'Manual':    'Manual',    'M':    'Manual',
}

df['transmission_clean'] = df['transmission'].map(TRANSMISSION)

# Anything that did not map becomes NaN — check before moving on.
print('unmapped:', df['transmission_clean'].isna().sum())
df['transmission_clean'].value_counts(dropna=False)


Using `.map()` with an explicit dictionary — rather than something clever — means anything
you did not anticipate turns into `NaN` and shows up in that check, instead of silently
passing through unchanged.

---

## 2.3  color — the missing values  🗣️ **CLASS DECISION**

First, the trap. pandas turns `""` and `N/A` into `NaN` on its own, but leaves
`unknown` and `-` as ordinary strings. So `isna()` under-reports.

Run both of these and watch the two numbers disagree.

In [ ]:
print('isna() says:', df['color'].isna().sum())
print()
print(df['color'].value_counts(dropna=False))


Make every sentinel mean the same thing to pandas, then count again.

In [ ]:
MISSING_SENTINELS = ['unknown', '-', 'n/a', 'N/A', '']

df['color_clean'] = (df['color'].astype('string').str.strip()
                     .replace(MISSING_SENTINELS, pd.NA))

print('really missing:', df['color_clean'].isna().sum(), 'of', len(df))
df['color_clean'].value_counts(dropna=False)


### Now the decision — this cell is deliberately empty

The room votes, and we implement whatever the room decided. The four options:

1. **Leave it as NaN** — the default, and usually right
2. **Drop those rows** — `df.dropna(subset=['color_clean'])`; count what you lose
3. **Drop the column** — only if there is almost nothing there
4. **Fill it** — with a literal `'Unknown'` category, never with a guess dressed up as data

Remember the question that flips the answer: *does color affect price?* If color is the
variable you are studying, you cannot invent it.

In [ ]:
# ── LIVE: implement the class decision here ──



**Write down what we chose and why:**

> _decision:_ 
>
> _because:_ 

---

## 2.4  Duplicates — two different problems  🗣️ **CLASS DECISION**

Start by proving that the obvious call does nothing.

In [ ]:
print('exact duplicate rows:', df.duplicated().sum())


Zero. So there are no duplicates, right? Run these two instead.

In [ ]:
dupe_ids = df[df.duplicated(subset='listing_id', keep=False)]
dupe_ids[['listing_id', 'make', 'model', 'year', 'price', 'price_usd']]


In [ ]:
compare_cols = ['make_clean', 'model', 'year_num', 'price_usd', 'mileage_mi', 'color_clean']
same_car = df[df.duplicated(subset=compare_cols, keep=False)]
same_car[['listing_id'] + compare_cols]


Two different problems wearing the same word:

- **Same `listing_id`, different price** — one listing, two conflicting records. A data-entry
  conflict. Which price is true? You cannot tell from this file alone.
- **Different `listing_id`, everything else identical** — probably one car posted twice.
  Or genuinely two identical cars on the same lot. Also undecidable from this file alone.

`drop_duplicates()` with no arguments catches neither, because it compares whole rows.

### The decision — deliberately empty again

In [ ]:
# ── LIVE: implement the class decision here ──
# Hints, if you want them:
#   df.drop_duplicates(subset='listing_id', keep='last')
#   df.sort_values('price_usd').drop_duplicates(subset='listing_id', keep='first')
#   df.drop_duplicates(subset=compare_cols, keep='first')

before = len(df)


rows_changed(before, len(df), 'de-duplication')


**Write down what we chose and why:**

> _decision:_ 
>
> _because:_ 

---

## 2.5  Impossible values — flag, do not delete  🗣️ **CLASS DECISION**

A `$1` Ford F-150 with `seller = Auction` is plausible. A `$0` listing is a placeholder.
A year of 2105 is a typo. Mileage of −4,500 is impossible.

We cannot phone anyone, so we look for a pattern in the file itself.

In [ ]:
df.loc[df['price_usd'] < 100, ['listing_id', 'make_clean', 'model', 'price', 'price_usd', 'seller']]


In [ ]:
# Is 'cheap' associated with 'Auction'? Check before you decide.
df.groupby('seller')['price_usd'].agg(['count', 'min', 'median'])


Rather than deleting suspicious rows, we mark them. A flagged row can be excluded later
by whoever needs to; a deleted row is gone and nobody downstream knows it existed.

In [ ]:
df['flag_price']   = df['price_usd'] < 100
df['flag_year']    = ~df['year_num'].between(1990, 2027)
df['flag_mileage'] = df['mileage_mi'] < 0

flag_cols = ['flag_price', 'flag_year', 'flag_mileage']
df['flag_any'] = df[flag_cols].any(axis=1)

print(df[flag_cols].sum())
print()
print('rows with at least one flag:', int(df['flag_any'].sum()), 'of', len(df))


In [ ]:
df.loc[df['flag_any'], ['listing_id', 'make_clean', 'model', 'year', 'price', 'mileage', 'seller']]


## 2.6  Load — the L in ETL

Keep the columns that are real variables, in a sensible order, and write them somewhere
durable. This file — not the notebook's memory — is what the next analysis reads.

In [ ]:
KEEP = ['listing_id', 'make_clean', 'model', 'year_num', 'price_usd',
        'mileage_mi', 'color_clean', 'transmission_clean', 'listed_on',
        'seller', 'flag_any']

clean = df[KEEP].rename(columns={
    'make_clean':         'make',
    'year_num':           'year',
    'price_usd':          'price',
    'mileage_mi':         'mileage',
    'color_clean':        'color',
    'transmission_clean': 'transmission',
})

clean['model'] = clean['model'].astype('string').str.strip()   # 'Corolla ' -> 'Corolla'

clean.info()


In [ ]:
OUT = Path('../data/clean/used_cars_clean.csv')
if not OUT.parent.exists():
    OUT = Path('used_cars_clean.csv')     # Colab fallback

OUT.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(OUT, index=False)

print('wrote', OUT.resolve())
print('raw :', raw.shape)
print('clean:', clean.shape)


---

## What we decided tonight

Fill this in before you close the notebook. In the midterm report this section is graded.

| # | Decision | Why |
|---|---|---|
| 1 | Dropped the `TOTAL` row | It is a spreadsheet footer, not an observation |
| 2 | Treated a missing mileage unit as miles | Every labelled non-`km` row in the file is `mi` |
| 3 | `Toyata` → `Toyota`, `Chevy` → `Chevrolet` | Same models and price range; typo and informal name |
| 4 | *(color)* | |
| 5 | *(duplicates)* | |
| 6 | Flagged impossible values instead of deleting | A flag is reversible; a delete is invisible |

---

### Before Monday

Redo this from `used_cars_raw.csv` in your own notebook, making your own calls.
Where you disagree with what we did in class is the interesting part — HW1 asks you
to write down three decisions and one sentence of why for each.